In [1]:

import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from scipy.sparse import save_npz
import joblib

df = pd.read_csv("hotel_bookings.csv")

print("Original Dataset Shape:", df.shape)

print("\nMissing Values Before Cleaning:")
print(df.isnull().sum())

print("\nStatistics Before Cleaning:")
print(df.describe(include="all"))

before = len(df)
df = df.drop_duplicates().copy()

print("\nDuplicates Removed:", before - len(df))

df = df.drop(columns=["company"])

df["agent"] = df["agent"].fillna(0).astype(int)
df["country"] = df["country"].fillna("Unknown")
df["children"] = df["children"].fillna(df["children"].median()).astype(int)

df["reservation_status_date"] = pd.to_datetime(
    df["reservation_status_date"],
    errors="coerce",
    dayfirst=True
)

df["arrival_date_month"] = df["arrival_date_month"].str.strip()

df["arrival_date_month_num"] = pd.to_datetime(
    df["arrival_date_month"],
    format="%B",
    errors="coerce"
).dt.month

df["arrival_date"] = pd.to_datetime(
    dict(
        year=df["arrival_date_year"],
        month=df["arrival_date_month_num"],
        day=df["arrival_date_day_of_month"]
    ),
    errors="coerce"
)

invalid_rows = (
    (df["adults"] + df["children"] + df["babies"] == 0) |
    (df["adr"] < 0)
)

print("\nInvalid Rows Removed:", invalid_rows.sum())

df = df.loc[~invalid_rows].copy()

outlier_columns = ["adr", "lead_time"]

for col in outlier_columns:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1

    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    outliers = (df[col] < lower) | (df[col] > upper)

    print(col, "Outliers:", outliers.sum())

    df[col] = df[col].clip(
        lower=max(0, lower),
        upper=upper
    )

df["total_nights"] = (
    df["stays_in_weekend_nights"] +
    df["stays_in_week_nights"]
)

df["total_guests"] = (
    df["adults"] +
    df["children"] +
    df["babies"]
)

df["is_family"] = (
    (df["children"] > 0) |
    (df["babies"] > 0)
).astype(int)

df["is_weekend_stay"] = (
    df["stays_in_weekend_nights"] > 0
).astype(int)

df["previous_booking_total"] = (
    df["previous_cancellations"] +
    df["previous_bookings_not_canceled"]
)

df["has_previous_cancellations"] = (
    df["previous_cancellations"] > 0
).astype(int)

df["has_special_requests"] = (
    df["total_of_special_requests"] > 0
).astype(int)

df["booking_value_estimate"] = (
    df["adr"] * df["total_nights"]
)

df["arrival_season"] = df["arrival_date_month_num"].map({
    12: "Winter",
    1: "Winter",
    2: "Winter",
    3: "Spring",
    4: "Spring",
    5: "Spring",
    6: "Summer",
    7: "Summer",
    8: "Summer",
    9: "Autumn",
    10: "Autumn",
    11: "Autumn"
})

df["lead_time_category"] = pd.cut(
    df["lead_time"],
    bins=[-1, 7, 30, 90, 180, np.inf],
    labels=[
        "Last minute",
        "Short term",
        "Medium term",
        "Long term",
        "Very long term"
    ]
)

df["stay_type"] = np.select(
    [
        (df["stays_in_weekend_nights"] > 0) &
        (df["stays_in_week_nights"] > 0),

        (df["stays_in_weekend_nights"] > 0) &
        (df["stays_in_week_nights"] == 0),

        (df["stays_in_weekend_nights"] == 0) &
        (df["stays_in_week_nights"] > 0)
    ],
    [
        "Weekend and Weekday",
        "Weekend Only",
        "Weekday Only"
    ],
    default="No Stay"
)

print("\nNew Features Created:")
print([
    "total_nights",
    "total_guests",
    "is_family",
    "is_weekend_stay",
    "previous_booking_total",
    "has_previous_cancellations",
    "has_special_requests",
    "booking_value_estimate",
    "arrival_season",
    "lead_time_category",
    "stay_type"
])

df["agent"] = df["agent"].astype(str)

cleaned = df.copy()

for col in cleaned.select_dtypes(
    include=["object", "category"]
).columns:
    cleaned[col] = (
        cleaned[col].astype("string").fillna("Unknown")
    )

for col in cleaned.select_dtypes(include=np.number).columns:
    cleaned[col] = cleaned[col].fillna(cleaned[col].median())

cleaned.to_csv("hotel_bookings_cleaned.csv", index=False)

print("\nMissing Values After Cleaning:")
print(cleaned.isnull().sum())

print("\nStatistics After Cleaning:")
print(cleaned.describe(include="all"))

print("\nFinal Dataset Shape:", cleaned.shape)

X = df.drop(columns=[
    "is_canceled",
    "reservation_status",
    "reservation_status_date",
    "arrival_date"
])

y = df["is_canceled"]

numeric_features = X.select_dtypes(
    include=np.number
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object", "category"]
).columns.tolist()

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("\nPipeline Completed Successfully!")

print("Training Data Shape:", X_train_processed.shape)
print("Testing Data Shape:", X_test_processed.shape)

save_npz(
    "hotel_train_preprocessed.npz",
    X_train_processed
)

save_npz(
    "hotel_test_preprocessed.npz",
    X_test_processed
)

joblib.dump(
    preprocessor,
    "hotel_preprocessing_pipeline.pkl"
)

pd.Series(
    preprocessor.get_feature_names_out()
).to_csv("feature_names.csv", index=False)

y_train.to_csv("y_train.csv", index=False)
y_test.to_csv("y_test.csv", index=False)

print("\nAll Files Saved Successfully!")


Original Dataset Shape: (74143, 32)

Missing Values Before Cleaning:
hotel                                 0
is_canceled                           0
lead_time                             0
arrival_date_year                     0
arrival_date_month                    0
arrival_date_week_number              0
arrival_date_day_of_month             0
stays_in_weekend_nights               0
stays_in_week_nights                  0
adults                                0
children                              4
babies                                0
meal                                  0
country                             486
market_segment                        0
distribution_channel                  0
is_repeated_guest                     0
previous_cancellations                0
previous_bookings_not_canceled        0
reserved_room_type                    0
assigned_room_type                    0
booking_changes                       0
deposit_type                          0
agent      